# Sales Agent Orchestrator using OpenAI Agent SDK

In [ ]:
# Imports
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio 

In [ ]:
load_dotenv(override=True)

In [ ]:
instructions1 = "You are a sales expert of a healthcare product company with deep clinical knowledge. \
You write a concise cold email to hospital CEO highlighting how our product improves patient outomes, \
backed by clinical evidence, efficiency gains and compliance benefits. Keep tone professional, data-driven and outcome-focused."

instructions2 = "You are a relationship-driven healthcare sales consultant. \
You write a warm, personalized cold email to healthcare organization CEO focusing on understanding their challenges, \
building trust and positioning our product as a long-term partner solution. Keep tone conversational and empathetic."

instructions3 = "You are results-oriented sales strategist. \
You write a sharp cold email to a healthcare CEO emphasizing Return on Investment(RoI), cost savings, \
operational efficiency and revenue growth enabled by our product. Keep the tone crisp and business-focused."

In [ ]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-oss-20b")

sales_agent2 = Agent(name="Empathetic Sales Agent", instructions=instructions2, model="gpt-oss-20b")

sales_agent3 = Agent(name="Business-focused Sales Agent", instructions=instructions3, model="gpt-oss-20b")

In [ ]:
# [Only for learning, no need to execute for sales agent orchestrator] 
# 1. Start the agent
# 2. Litens to streaming outputs
# 3. Extrat texts
# 4. Print the text instantly  
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

In [ ]:
#[Only for learning, no need to execute for sales agent orchestrator] 
# Equivalent non-streaming version of above code
result = await Runner.run(sales_agent1, input="Write a cold sales email")
print(result.final_output)

In [ ]:
message = "Write a cold sales email for CEO of hospitals"

with trace("Parallel execution of all sales agent's cold emails"):
    results = await asyncio.gather(
     Runner.run(sales_agent1, message),
     Runner.run(sales_agent2, message),
     Runner.run(sales_agent3, message),   
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

In [ ]:
manager_instructions = "You are a sales manager of the healthcare product company. \
You have to choose the best cold sales email from the given options. \
Act as a client and pick one you are most likely to respond to. \
You don't have to give explanation, just reply with selected email."

sales_manager = Agent(
    name="sales_manager",
    instructions=manager_instructions,
    model="gpt-oss-20b"
)

In [ ]:
all_emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

best_email = await Runner.run(sales_manager, all_emails)

print(f"Best sales email:\n{best_email.final_output}")